# Project Virtual Painter

Find the color from webcam and place points continuously


##### Stack images

In [30]:
# stack images (detailed explanation ):
# https://www.geeksforgeeks.org/stacking-images-in-python-using-numpy/
def stackImages(scale,imgArray):
    rows = len(imgArray)
    cols = len(imgArray[0])
    rowsAvailable = isinstance(imgArray[0], list)
    width = imgArray[0][0].shape[1]
    height = imgArray[0][0].shape[0]
    if rowsAvailable :
        for x in range(0,rows):
            for y in range (0,cols):
                if imgArray[x][y].shape[:2] == imgArray[0][0].shape[:2]:
                    imgArray[x][y] = cv2.resize(imgArray[x][y], (0,0), None, scale, scale)
                else:
                    imgArray[x][y] = cv2.resize(imgArray[x][y], (imgArray[0][0].shape[1], imgArray[0][0].shape[0]), None, scale, scale)
                if len(imgArray[x][y].shape) == 2: imgArray[x][y]= cv2.cvtColor(imgArray[x][y], cv2.COLOR_GRAY2BGR)
        imageBlank = np.zeros((height, width, 3), np.uint8)
        hor = [imageBlank]*rows
        hor_con = [imageBlank]*rows
        for x in range(0,rows):
            hor[x] = np.hstack(imgArray[x])
        ver = np.vstack(hor)
    else:
        for x in range (0,rows):
            if imgArray[x].shape[:2] == imgArray[0].shape[:2]:
                imgArray[x] = cv2.resize(imgArray[x], (0,0), None, scale, scale)
            else:
                imgArray[x] = cv2.resize(imgArray[x], (imgArray[0].shape[1], imgArray[0].shape[0]), None, scale, scale)
            if len(imgArray[x].shape) == 2: imgArray[x] = cv2.cvtColor(imgArray[x], cv2.COLOR_GRAY2BGR)
        hor= np.hstack(imgArray)
        ver = hor
    return ver


##### Color picker program

In [33]:
import cv2
import numpy as np
frameWidth = 640
frameHeight = 480
cap = cv2.VideoCapture(0)
cap.set(3,frameWidth)
cap.set(4,frameHeight)

def empty(a):
    pass

# Create window and trackbars
cv2.namedWindow("HSV")
cv2.resizeWindow("HSV",640,240)
cv2.createTrackbar("HUE Min","HSV",0,179,empty)
cv2.createTrackbar("HUE Max","HSV",179,179,empty)
cv2.createTrackbar("SAT Min","HSV",0,255,empty)
cv2.createTrackbar("SAT Max","HSV",255,255,empty)
cv2.createTrackbar("VALUE Min","HSV",0,255,empty)
cv2.createTrackbar("VALUE Max","HSV",255,255,empty)

try:
    while True:
        success, img = cap.read()
        if not success:
            print("Failed to grab frame")
            break
            
        imgHSV = cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
        h_min = cv2.getTrackbarPos("HUE Min","HSV")
        h_max = cv2.getTrackbarPos("HUE Max","HSV")
        s_min = cv2.getTrackbarPos("SAT Min","HSV")
        s_max = cv2.getTrackbarPos("SAT Max","HSV")
        v_min = cv2.getTrackbarPos("VALUE Min","HSV")
        v_max = cv2.getTrackbarPos("VALUE Max","HSV")
        
        lower = np.array([h_min,s_min,v_min])
        upper = np.array([h_max,s_max,v_max])
        mask = cv2.inRange(imgHSV,lower,upper)
        imgResult = cv2.bitwise_and(img,img,mask=mask)
        imgStack = stackImages(0.25, ([img, imgHSV, mask,imgResult]))
        cv2.imshow('Stacked Images', imgStack)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    # Proper cleanup
    print("Cleaning up...")
    cap.release()
    cv2.destroyAllWindows()
    # For Windows, sometimes we need multiple calls
    for i in range(4):
        cv2.waitKey(1)

Cleaning up...


In [35]:
import time
import cv2
import numpy as np
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.8,
    min_tracking_confidence=0.8
)

frameWidth = 640
frameHeight = 480
cap = cv2.VideoCapture(0)
cap.set(3, frameWidth)
cap.set(4, frameHeight)
cap.set(10, 150)

myColors = [[143,50,141,165,173,255]]
myColorValues = [[226,43,138]]
pointSize = 5

# ---- drawing / smoothing state ----
canvas = np.zeros((frameHeight, frameWidth, 3), dtype=np.uint8)
last_points = {i: None for i in range(len(myColors))}
smoothed_points = {i: None for i in range(len(myColors))}
LINE_THICKNESS = 8
SMOOTHING_ENABLED = True
SMOOTHING_ALPHA = 0.6

# ---- no-hands erase settings ----
NO_HANDS_HOLD = 3.0     # seconds without hands before erasing (set 0.0 for immediate)
last_nohands_time = None
last_erase_time = 0.0
ERASE_COOLDOWN = 0.5    # short cooldown after erase to avoid immediate retrigger

def getContours(img_mask):
    contours, _ = cv2.findContours(img_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    best_area = 0
    tip_x, tip_y = 0, 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 200 and area > best_area:
            best_area = area
            pts = cnt.reshape(-1, 2)
            min_idx = np.argmin(pts[:, 1])  # topmost point
            tip_x, tip_y = int(pts[min_idx, 0]), int(pts[min_idx, 1])
    return tip_x, tip_y

def check_left_hand_open(results, img_w):
    """Return True if a visual-left hand is detected and its four fingers (not thumb) are open."""
    if not results.multi_hand_landmarks:
        return False
    for hand_landmarks in results.multi_hand_landmarks:
        xs = [lm.x for lm in hand_landmarks.landmark]
        avg_x = sum(xs) / len(xs)
        pixel_x = avg_x * img_w
        visual_left = pixel_x < (img_w / 2)
        finger_tips = [8, 12, 16, 20]
        is_open = all(
            hand_landmarks.landmark[tip].y < hand_landmarks.landmark[tip - 2].y
            for tip in finger_tips
        )
        if visual_left and is_open:
            return True
    return False

def findColor(img, myColors):
    imgHSV = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    count = 0
    newPoints = []
    for color in myColors:
        lower = np.array(color[0:3])
        upper = np.array(color[3:6])
        mask = cv2.inRange(imgHSV, lower, upper)
        x, y = getContours(mask)
        if x != 0 and y != 0:
            newPoints.append([x, y, count])
        count += 1
    return newPoints

try:
    while True:
        success, img = cap.read()
        if not success:
            break

        img = cv2.flip(img, 1)  # mirror
        imgResult = img.copy()

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        # ----- erase when no hands present for NO_HANDS_HOLD seconds -----
        now = time.time()
        hands_visible = bool(results.multi_hand_landmarks)
        # start or reset the no-hands timer
        if not hands_visible:
            if last_nohands_time is None:
                last_nohands_time = now
            elapsed = now - last_nohands_time
            remaining = max(0.0, NO_HANDS_HOLD - elapsed)
            # show countdown if hold time > 0
            if NO_HANDS_HOLD > 0:
                cv2.putText(imgResult, f"No hands: erase in {remaining:.1f}s", (10,60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
            if elapsed >= NO_HANDS_HOLD and (now - last_erase_time) > ERASE_COOLDOWN:
                # erase: clear canvas and reset buffers
                canvas.fill(0)
                for k in last_points:
                    last_points[k] = None
                    smoothed_points[k] = None
                last_erase_time = now
                last_nohands_time = None
                # feedback: quick flash
                flash = imgResult.copy()
                flash[:] = 255
                imgResult = cv2.addWeighted(flash, 0.35, imgResult, 0.65, 0)
                cv2.putText(imgResult, "ERASED (no hands)", (frameWidth//2 - 140, frameHeight//2),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 3)
        else:
            # hands visible -> reset timer
            last_nohands_time = None

        # ----- drawing permission: only if visual-left hand is open -----
        can_draw = check_left_hand_open(results, img.shape[1])
        if not can_draw:
            # reset last points so lines don't connect across pauses
            for k in last_points:
                last_points[k] = None
                smoothed_points[k] = None

        # ----- find colored tip and draw continuous smoothed lines -----
        newPoints = findColor(img, myColors) if can_draw else []
        for x, y, colorId in newPoints:
            cur = np.array([x, y], dtype=float)
            if SMOOTHING_ENABLED:
                prev_s = smoothed_points[colorId]
                if prev_s is None:
                    smooth = cur
                else:
                    smooth = SMOOTHING_ALPHA * prev_s + (1.0 - SMOOTHING_ALPHA) * cur
            else:
                smooth = cur

            prev_pt = last_points[colorId]
            if prev_pt is not None:
                pt1 = (int(prev_pt[0]), int(prev_pt[1]))
                pt2 = (int(smooth[0]), int(smooth[1]))
                cv2.line(canvas, pt1, pt2, myColorValues[colorId], LINE_THICKNESS, lineType=cv2.LINE_AA)

            last_points[colorId] = smooth.copy()
            smoothed_points[colorId] = smooth.copy()

        # ----- dim background to 30% and overlay opaque canvas -----
        dimmed = cv2.convertScaleAbs(imgResult, alpha=0.3, beta=0)
        imgResult = cv2.add(dimmed, canvas)

        # small HUD
        cv2.putText(imgResult, f"Hands visible: {'YES' if hands_visible else 'NO'}", (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0) if hands_visible else (0,0,255), 2)

        cv2.imshow("Result", imgResult)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    hands.close()
    cap.release()
    cv2.destroyAllWindows()
    for i in range(4):
        cv2.waitKey(1)
